In [2]:
import random
from pathlib import Path

import sqlite3

import pandas as pd
import sklearn as sk
import numpy as np
from scipy import stats

PROJECT_ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = PROJECT_ROOT / "data" / "fifa-2026"
DB_PATH = PROJECT_ROOT / "db" / "fifa2026.db"
DB_PATH.parent.mkdir(exist_ok=True)

PERMA_SEED = 42
random.seed(PERMA_SEED)
np.random.seed(PERMA_SEED)

In [4]:
def q(sql, params=None):
    with sqlite3.connect(DB_PATH) as c:
        return pd.read_sql_query(sql, c, params=params)

In [5]:
final_dataset = q("""
    WITH
squad_totals AS (
    SELECT
        team_id,
        SUM(market_value_eur) / 1e6 AS squad_value_m,
        AVG((julianday('2026-06-11') - julianday(date_of_birth)) / 365.25) AS squad_age,
        SUM(caps) AS total_caps,
        AVG(height_cm) AS avg_height,
        SUM(goals) AS goals_sum
    FROM squads_and_players
    GROUP BY team_id
),
match_multiplier AS (
    SELECT match_id, date, home_team_id AS team_id FROM matches
    UNION ALL
    SELECT match_id, date, away_team_id FROM matches
),
rest AS (
    SELECT
        match_id,
        team_id,
        julianday(date) - julianday(LAG(date) OVER (PARTITION BY team_id ORDER BY date, match_id)) AS rest_days
    FROM match_multiplier
)
SELECT
    matches.match_id,
    matches.home_team_id,
    matches.away_team_id,
    t1.fifa_ranking_pre_tournament - t2.fifa_ranking_pre_tournament AS ranking_difference,
    t1.elo_rating - t2.elo_rating AS elo_rating_difference,
    home.squad_value_m - away.squad_value_m AS squad_value_difference,
    home.squad_age - away.squad_age AS squad_age_difference,
    home.total_caps - away.total_caps AS caps_difference,
    home.avg_height - away.avg_height AS height_difference,
    home.goals_sum - away.goals_sum AS goal_total_difference,
    COALESCE(rest_home.rest_days, 10) - COALESCE(rest_away.rest_days, 10) AS rest_days_difference,
    matches.home_score - matches.away_score AS goal_difference
FROM matches
JOIN teams AS t1 ON t1.team_id = matches.home_team_id
JOIN teams AS t2 ON t2.team_id = matches.away_team_id
JOIN squad_totals AS home ON home.team_id = matches.home_team_id
JOIN squad_totals AS away ON away.team_id = matches.away_team_id
JOIN rest AS rest_home ON rest_home.match_id = matches.match_id AND rest_home.team_id = matches.home_team_id
JOIN rest AS rest_away ON rest_away.match_id = matches.match_id AND rest_away.team_id = matches.away_team_id
ORDER BY matches.match_id;
""")

In [8]:
>>> len(final_dataset)

104